In [1]:
import pandas as pd
import numpy as np

# Загрузим данные
actions = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\Python Analytics\actions_log.csv', parse_dates=["action_dttm"])
snapshots = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\Python Analytics\daily_snapshot.csv', parse_dates=["snapshot_date"])

# Обнулим время, оставив только дату
actions["action_date"] = actions["action_dttm"].dt.normalize()
snapshots["snapshot_date"] = snapshots["snapshot_date"].dt.normalize()

In [2]:
actions.head()

,customer_id,action_dttm,action_type,action_amount,action_date
0,400000,2026-01-01 14:59:00,payment,13309.31,2026-01-01
1,400000,2026-01-01 19:11:00,call,0.00,2026-01-01
2,400000,2026-01-08 14:37:00,payment,6772.92,2026-01-08
3,400000,2026-01-10 08:04:00,call,0.00,2026-01-10
4,400000,2026-01-12 14:06:00,payment,8731.15,2026-01-12


In [3]:
snapshots.head()

,customer_id,snapshot_date,dpd,outstanding_amount
0,400000,2026-03-11,10,19000.14
1,400000,2026-03-15,22,16914.99
2,400001,2026-03-30,26,16142.60
3,400001,2026-04-03,33,16326.84
4,400002,2026-02-21,90,44636.91


In [4]:
# Соберем данные по дням и по клиентам, добавив необходимые столбцы
daily = (actions.assign(calls=(actions["action_type"] == "call").astype(int), sms=(actions["action_type"] == "sms").astype(int),
        promises=(actions["action_type"] == "promise").astype(int), payments_amount=np.where(actions["action_type"] == "payment", actions["action_amount"], 0.0),
        promised_amount=np.where(actions["action_type"] == "promise", actions["action_amount"], 0.0))
        .groupby(["customer_id", "action_date"], as_index=False)[["calls", "sms", "promises", "payments_amount", "promised_amount"]]
        .sum()
        .sort_values(["customer_id", "action_date"]))

# Посчитаем кумулятивные суммы по каждому клиенту
for col in ["calls", "sms", "promises", "payments_amount", "promised_amount"]:
    daily[f"cum_{col}"] = daily.groupby("customer_id")[col].cumsum()

In [5]:
# Создадим функцию для оконного признака

# Считает сумму за [snapshot_date - days, snapshot_date)
def add_window_feature(result, daily, base_col, feature_name, days):
    pieces = []

    # Работаем с каждой колонкой отдельно
    daily_sub = daily[["customer_id", "action_date", f"cum_{base_col}"]].copy()

    # Идем по клиентам
    for customer_id, snap_grp in result.groupby("customer_id", sort=False):
        # Сортируем данные о клиентах
        snap_grp = snap_grp.sort_values("snapshot_date").copy()
        act_grp = daily_sub[daily_sub["customer_id"] == customer_id].sort_values("action_date").copy()

        #Создаем границы окна
        snap_grp["right"] = snap_grp["snapshot_date"] - pd.Timedelta(days=1)
        snap_grp["left"] = snap_grp["snapshot_date"] - pd.Timedelta(days=days + 1)

        # Посчитаем значения за обозначеный период
        if act_grp.empty:
            snap_grp[feature_name] = 0
        else:
            right = pd.merge_asof(snap_grp[["snapshot_date", "right"]].sort_values("right"),
                                  act_grp[["action_date", f"cum_{base_col}"]].sort_values("action_date"), 
                                  left_on="right",
                                  right_on="action_date",
                                  direction="backward")[f"cum_{base_col}"].fillna(0).to_numpy()

            left = pd.merge_asof(snap_grp[["snapshot_date", "left"]].sort_values("left"),
                                 act_grp[["action_date", f"cum_{base_col}"]].sort_values("action_date"),
                                 left_on="left",
                                 right_on="action_date",
                                 direction="backward")[f"cum_{base_col}"].fillna(0).to_numpy()

            snap_grp[feature_name] = right - left
            
        # Сохраним результаты по каждому клиенту
        pieces.append(snap_grp[[feature_name]])
        
    #Объеденим все данные по клиентам в одну таблицу
    result[feature_name] = pd.concat(pieces).sort_index()[feature_name].to_numpy()

In [6]:
# Создадим витрину

feature_df = snapshots.copy()

# Зададим необходимый список признаков
feature_specs = [("calls", "calls_last_7d", 7), ("sms", "sms_last_7d", 7), ("promises", "promises_last_30d", 30),
                 ("payments_amount", "payments_amount_last_30d", 30), ("promised_amount", "promised_amount_last_30d", 30)]

for base_col, feature_name, days in feature_specs:
    add_window_feature(feature_df, daily, base_col, feature_name, days)

In [7]:
# Вычислим кол-во дней с момента последнего платежа

# Оставим только платежи
payments = (
    actions.loc[actions["action_type"] == "payment", ["customer_id", "action_dttm"]]
    .sort_values(["customer_id", "action_dttm"])
)

days_since_parts = []

for customer_id, snap_grp in feature_df.groupby("customer_id", sort=False):
    # Отсортируем снапшоты и платежи
    snap_grp = snap_grp.sort_values("snapshot_date").copy()
    pay_grp = payments[payments["customer_id"] == customer_id].sort_values("action_dttm").copy()

    # Найдем последний платеж до момента снятия снапшота
    if pay_grp.empty:
        snap_grp["days_since_last_payment"] = np.nan
    else:
        merged = pd.merge_asof(snap_grp[["snapshot_date"]].sort_values("snapshot_date"), 
                               pay_grp[["action_dttm"]].sort_values("action_dttm"),
                               left_on="snapshot_date",
                               right_on="action_dttm",
                               direction="backward",
                               allow_exact_matches=False)

        # Посчтиаем разницу в днях
        snap_grp["days_since_last_payment"] = (snap_grp["snapshot_date"] - merged["action_dttm"].dt.normalize()).dt.days

    # Сохраняем результат по каждому клиенту
    days_since_parts.append(snap_grp[["days_since_last_payment"]])

# Объединим данные по всем клиентам
feature_df["days_since_last_payment"] = (pd.concat(days_since_parts).sort_index()["days_since_last_payment"].to_numpy())

# Приведем тип данных к int
for col in ["calls_last_7d", "sms_last_7d", "promises_last_30d"]:
    feature_df[col] = feature_df[col].fillna(0).astype(int)

# Приведем тип данных к float
for col in ["payments_amount_last_30d", "promised_amount_last_30d"]:
    feature_df[col] = feature_df[col].fillna(0.0)

# Посчитаем общее число контактов за последние 7 дней
feature_df["contacts_last_7d"] = feature_df["calls_last_7d"] + feature_df["sms_last_7d"]


In [8]:
# Найдем топ-20 клиентов с максимальным числом контактов и нулевой суммой оплат за последние 30 дней

# Отфильтруем клиентов, оставив только без платежей за последние 30 дней
top20 = (feature_df.query("payments_amount_last_30d == 0")
         .sort_values(["contacts_last_7d", "customer_id", "snapshot_date"], ascending=[False, True, True])
         .head(20))

print("Итоговая витрина:")
print(feature_df.head())

print("\nТоп-20 клиентов:")
print(top20[["customer_id", "snapshot_date", "contacts_last_7d", "calls_last_7d", "sms_last_7d", 
             "payments_amount_last_30d", "promises_last_30d", "promised_amount_last_30d", "days_since_last_payment"]])

Итоговая витрина:
   customer_id snapshot_date  dpd  outstanding_amount  calls_last_7d  \
0       400000    2026-03-11   10            19000.14              0   
1       400000    2026-03-15   22            16914.99              0   
2       400001    2026-03-30   26            16142.60              0   
3       400001    2026-04-03   33            16326.84              0   
4       400002    2026-02-21   90            44636.91              1   

   sms_last_7d  promises_last_30d  payments_amount_last_30d  \
0            0                  0                      0.00   
1            0                  0                      0.00   
2            1                  1                  21487.22   
3            0                  1                  21487.22   
4            0                  0                      0.00   

   promised_amount_last_30d  days_since_last_payment  contacts_last_7d  
0                      0.00                     58.0                 0  
1                      0